# sphinx_autodoc_typehints

sphinx-autodoc-typehints 3.17.5 reads the type annotations of what `sphinx.ext.autodoc` documents and writes them into
the page: each parameter's type beside its description, and the return type, so docstrings need not repeat them. These
cells build small Sphinx projects in a temporary folder and read the HTML: the default rendering, an undocumented
parameter, defaults shown beside the type, the return type switched off, and a strict build over a generic type.

In [ ]:
# Example 1: the types come from the annotations alone; the docstring only describes the parameters.
import pathlib, re, sys, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "geo.py").write_text('def area(w: float, h: int) -> float:\n    """Area.\n\n    :param w: width\n    :param h: height\n    """\n    return w * h\n')
(src / "conf.py").write_text(f"import sys\nsys.path.insert(0, {str(src)!r})\nextensions = ['sphinx.ext.autodoc', 'sphinx_autodoc_typehints']\n")
(src / "index.rst").write_text("API\n===\n\n.. automodule:: geo\n   :members:\n")
rc = build_main(["-q", "-W", "-n", "-b", "html", str(src), str(src / "_out")])
text = " ".join(re.sub(r"<[^>]+>", " ", (src / "_out" / "index.html").read_text()).split())
print("rc", rc)
print(text[text.find("Parameters"):][:160])
assert rc == 0
assert "w ( float ) – width" in text and "h ( int ) – height" in text
assert "Return type : float" in text

In [ ]:
# Example 2: always_document_param_types lists a parameter the docstring never mentions, with its type.
import pathlib, re, sys, tempfile
from sphinx.cmd.build import build_main

def build(extra_conf):
    src = pathlib.Path(tempfile.mkdtemp())
    (src / "paramsmod.py").write_text('def f(a: str, b: bytes) -> None:\n    """Do it.\n\n    :param a: the text\n    """\n')
    (src / "conf.py").write_text(f"import sys\nsys.path.insert(0, {str(src)!r})\nextensions = ['sphinx.ext.autodoc', 'sphinx_autodoc_typehints']\n" + extra_conf)
    (src / "index.rst").write_text("API\n===\n\n.. autofunction:: paramsmod.f\n")
    assert build_main(["-q", "-W", "-b", "html", str(src), str(src / "_out")]) == 0
    return " ".join(re.sub(r"<[^>]+>", " ", (src / "_out" / "index.html").read_text()).split())

plain, always = build(""), build("always_document_param_types = True\n")
print(always[always.find("Parameters"):][:120])
assert "b ( bytes )" not in plain
assert "a ( str ) – the text" in always and "b ( bytes )" in always

In [ ]:
# Example 3: typehints_defaults = "comma" writes each default value beside the parameter's type.
import pathlib, re, sys, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "retrymod.py").write_text('def retry(times: int = 3, delay: float = 0.5) -> None:\n    """Retry.\n\n    :param times: attempts\n    :param delay: seconds between attempts\n    """\n')
(src / "conf.py").write_text(f"import sys\nsys.path.insert(0, {str(src)!r})\nextensions = ['sphinx.ext.autodoc', 'sphinx_autodoc_typehints']\ntypehints_defaults = 'comma'\n")
(src / "index.rst").write_text("API\n===\n\n.. autofunction:: retrymod.retry\n")
assert build_main(["-q", "-W", "-b", "html", str(src), str(src / "_out")]) == 0
text = " ".join(re.sub(r"<[^>]+>", " ", (src / "_out" / "index.html").read_text()).split())
print(text[text.find("Parameters"):][:200])
assert "times ( int , default: 3 ) – attempts" in text
assert "delay ( float , default: 0.5 )" in text

In [ ]:
# Example 4: typehints_document_rtype = False drops the "Return type" field the extension adds by default.
import pathlib, re, sys, tempfile
from sphinx.cmd.build import build_main

def rtype_shown(flag):
    src = pathlib.Path(tempfile.mkdtemp())
    (src / f"namemod_{flag}.py").write_text('def name() -> str:\n    """The name."""\n    return "x"\n')
    (src / "conf.py").write_text(f"import sys\nsys.path.insert(0, {str(src)!r})\nextensions = ['sphinx.ext.autodoc', 'sphinx_autodoc_typehints']\ntypehints_document_rtype = {flag}\n")
    (src / "index.rst").write_text(f"API\n===\n\n.. autofunction:: namemod_{flag}.name\n")
    assert build_main(["-q", "-W", "-b", "html", str(src), str(src / "_out")]) == 0
    return "Return type" in (src / "_out" / "index.html").read_text()

print("on:", rtype_shown(True), "off:", rtype_shown(False))
assert rtype_shown(True) is True
assert rtype_shown(False) is False

In [ ]:
# Example 5: a strict (-W -n) build of a class whose method takes and returns generic and optional types.
import pathlib, re, sys, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "store.py").write_text(
    'class Store:\n    """A store."""\n\n'
    '    def get(self, key: str, default: int | None = None) -> dict[str, list[int]]:\n'
    '        """Look up.\n\n        :param key: the key\n        :param default: used when missing\n        """\n        return {}\n')
(src / "conf.py").write_text(f"import sys\nsys.path.insert(0, {str(src)!r})\nextensions = ['sphinx.ext.autodoc', 'sphinx_autodoc_typehints']\n")
(src / "index.rst").write_text("API\n===\n\n.. autoclass:: store.Store\n   :members:\n")
rc = build_main(["-q", "-W", "-n", "-b", "html", str(src), str(src / "_out")])
text = " ".join(re.sub(r"<[^>]+>", " ", (src / "_out" / "index.html").read_text()).split())
print("rc", rc, "|", text[text.find("Parameters"):][:200])
assert rc == 0
assert "key ( str ) – the key" in text
assert "dict[str,list[int]]" in text.replace(" ", "")